# 4. 大模型配合工具调用

## 4.1 模拟工具调用

In [ ]:
from random import randint
from typing import Annotated, Literal
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    if topic == "科技":
        return "最新科技新闻：AI 技术正在快速发展。"
    elif topic == "体育":
        return "最新体育新闻：中国队在比赛中取得了胜利。"
    else:
        return "最新娱乐新闻：吴亦凡发布了新专辑。"


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    user_input: str
    final_answer: str


def input_node(state: ChatState) -> dict:
    return {"messages": [HumanMessage(content=state["user_input"])]}


def llm_node(state: ChatState) -> Command[Literal["tool_node", "output_node"]]:
    ai_msg = model_with_tools.invoke(state["messages"])

    if ai_msg.tool_calls:
        next_node = "tool_node"
    else:
        next_node = "output_node"

    return Command(goto=next_node, update={"messages": [ai_msg]})


def tool_node(state: ChatState) -> dict:
    messages = state["messages"]
    last_msg = messages[-1]
    tool_calls = last_msg.tool_calls

    fail_prob = 6

    for call in tool_calls:
        if call["name"] == "get_weather":
            if randint(0, 9) < fail_prob:
                messages.append(
                    ToolMessage(
                        content="查询天气失败，请稍后再试", tool_call_id=call["id"]
                    )
                )
            else:
                result = get_weather.invoke(call["args"])
                messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

        elif call["name"] == "get_news":
            if randint(0, 9) < fail_prob:
                messages.append(
                    ToolMessage(
                        content="查询新闻失败，请稍后再试", tool_call_id=call["id"]
                    )
                )
            else:
                result = get_news.invoke(call["args"])
                messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

    return {"messages": messages}


def output_node(state: ChatState) -> dict:
    return {"final_answer": state["messages"][-1].content}


builder = StateGraph(state_schema=ChatState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()

res = graph.invoke(
    {
        "user_input": "帮我查询一下上海的天气和娱乐新闻",
        "messages": [SystemMessage("如果工具调用失败，必须重新调用直到成功为止")],
    }
)
rprint(res)

## 4.2 reda、write、shell 工具调用

In [ ]:
from typing import Literal
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import Command
from rich import print as rprint
import subprocess
from pathlib import Path
from tavily import TavilyClient

load_dotenv()

_tavily = TavilyClient()
model = ChatDeepSeek(model="deepseek-v4-flash")


@tool(parse_docstring=True)
def read(file_path: str) -> str:
    r"""读取本地文本文件的完整内容，以 UTF-8 编码返回。

    适合读取代码、配置、文档等文本文件。不支持二进制文件（如图片、Excel）。
    文件不存在或读取失败时，返回以"读取失败:"开头的错误信息，而非抛出异常。

    Args:
        file_path: 文件路径，支持绝对路径（如 D:/Work/project/main.py）
            或相对于工作目录的相对路径（如 src/config.json）。分隔符用 / 或 \ 均可。

    Returns:
        文件全文内容；失败时返回"读取失败: <原因>"。
    """
    try:
        return Path(file_path).read_text(encoding="utf-8")
    except Exception as e:
        return f"读取失败: {e}"


@tool(parse_docstring=True)
def write(file_path: str, content: str) -> str:
    """将文本内容写入本地文件（UTF-8 编码），会覆盖文件原有内容。

    父目录不存在时会自动创建。适合保存代码、配置、笔记等文本文件。

    Args:
        file_path: 目标文件路径，支持绝对或相对路径，如 D:/Work/demo/output.txt。
        content: 要写入的完整文本内容，为空字符串时会清空文件。

    Returns:
        成功返回"写入成功: <文件路径>"；失败返回"写入失败: <原因>"。
    """
    try:
        Path(file_path).parent.mkdir(parents=True, exist_ok=True)
        Path(file_path).write_text(content, encoding="utf-8")
        return f"写入成功: {file_path}"
    except Exception as e:
        return f"写入失败: {e}"


@tool(parse_docstring=True)
def shell(command: str) -> str:
    """在 Windows 上执行 PowerShell 命令并返回输出结果，可运行任何命令行操作。

    例如：查看目录（Get-ChildItem）、运行 Python 脚本（python main.py）、
    安装依赖（pip install requests）、Git 操作（git status）等。
    命令超时时间为 30 秒，长时间运行的命令（如训练模型）请勿使用此工具。
    路径中的反斜杠会自动转换为正斜杠，无需额外处理。

    Args:
        command: 要执行的 PowerShell 命令，如 "Get-ChildItem D:/Work" 或
            "python -m pytest tests/ -v"。多个命令用 ; 分隔。

    Returns:
        成功时返回命令的标准输出（stdout），无输出时返回"(无输出)"；
        失败返回"错误: <stderr>"；超过 30 秒返回"命令执行超时"。
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", command],
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode == 0:
            return result.stdout.strip() or "(无输出)"
        error = result.stderr.strip() or result.stdout.strip() or "未知错误"
        return f"错误: {error}"
    except subprocess.TimeoutExpired:
        return "命令执行超时"
    except Exception as e:
        return f"执行失败: {e}"


@tool(parse_docstring=True)
def web_search(query: str, max_results: int = 5) -> str:
    r"""联网搜索最新信息，返回与查询相关的网页标题、链接和内容摘要。

    适合查询实时或模型不知道的信息（新闻、最新版本、价格、文档等）。
    搜索失败时返回以"搜索失败:"开头的错误信息，而非抛出异常。

    Args:
        query: 搜索关键词或自然语言问题，如 "LangGraph 最新版本"。
        max_results: 返回结果数量，默认 5，建议 3~10。

    Returns:
        每条结果包含标题、链接和内容摘要的文本；失败时返回"搜索失败: <原因>"。
    """
    try:
        max_results = max(1, min(max_results, 10))
        res = _tavily.search(
            query=query, max_results=max_results, include_answer="basic"
        )
        parts = []
        if res.get("answer"):
            parts.append(f"参考答案: {res['answer']}")
        for r in res.get("results", []):
            parts.append(f"- {r['title']}\n  {r['url']}\n  {r['content']}")
        return "\n\n".join(parts) or "未找到相关结果"
    except Exception as e:
        return f"搜索失败: {e}"


@tool(parse_docstring=True)
def edit(file_path: str, old_text: str, new_text: str) -> str:
    r"""通过精确文本替换编辑文件，将 old_text 在文件中唯一匹配处替换为 new_text。

    适合对已有文件做局部修改（改一行、改函数名、改配置项等），不会动文件其他内容。
    old_text 必须与文件内容完全一致（包括缩进和换行）且在文件中唯一出现，
    不唯一时应扩大上下文范围使其唯一。一次只替换一处，修改多处请多次调用。
    文件不存在或编辑失败时，返回以"编辑失败:"开头的错误信息，而非抛出异常。

    Args:
        file_path: 目标文件路径，支持绝对或相对路径，如 D:/Work/project/main.py。
        old_text: 要替换的原文，必须与文件内容完全一致（含缩进和换行），
            如 "def add(a, b):\n    return a + b"。
        new_text: 替换后的新文本；传空字符串表示删除 old_text。

    Returns:
        成功返回"编辑成功: <文件路径>（替换 1 处）"；
        失败返回"编辑失败: <原因>"（含未找到、多处出现等具体提示）。
    """
    try:
        if not old_text:
            return "编辑失败: old_text 不能为空"

        content = Path(file_path).read_text(encoding="utf-8")

        count = content.count(old_text)
        if count == 0:
            return (
                "编辑失败: 未找到要替换的文本，old_text 必须与文件内容"
                "完全一致（包括缩进和换行），可先用 read 工具查看原文"
            )
        if count > 1:
            return (
                f"编辑失败: 要替换的文本出现 {count} 次，请扩大 old_text "
                "的上下文范围使其在文件中唯一"
            )
        if old_text == new_text:
            return "编辑失败: old_text 与 new_text 相同，文件不会有任何变化"

        Path(file_path).write_text(
            content.replace(old_text, new_text, 1), encoding="utf-8"
        )
        return f"编辑成功: {file_path}（替换 1 处）"
    except Exception as e:
        return f"编辑失败: {e}"


# 使用示例
tools = [read, write, edit, shell, web_search]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


def llm_node(state) -> Command[Literal["tool_node", END]]:  # type: ignore
    ai_msg = model_with_tools.invoke(state["messages"])
    if ai_msg.tool_calls:
        return Command(goto="tool_node", update={"messages": [ai_msg]})
    return Command(goto=END, update={"messages": [ai_msg]})


def tool_node(state: ChatState) -> Command[Literal["llm_node"]]:
    tool_map = {tool_.name: tool_ for tool_ in tools}
    tool_messages = []
    for call in state["messages"][-1].tool_calls:  # type: ignore
        selected_tool = tool_map.get(call["name"])
        if selected_tool is None:
            result = f"工具调用失败: 未知工具 {call['name']}"
        else:
            result = selected_tool.invoke(call["args"])
        tool_messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return Command(goto="llm_node", update={"messages": tool_messages})


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_node")

graph = builder.compile()
display(graph)
res = graph.invoke(
    {
        "messages": [
            SystemMessage(content="如果工具调用失败，必须根据失败结果重新制定策略"),
            HumanMessage(
                content="帮我在桌面创建一个科幻风格的网页，内容关于AI Agent。你可以先去调研一下这种网页的最佳布局再动手"
            ),
        ],
    }  # type: ignore
)
rprint(res)